# Assignment 4 — Hypothesis Testing with t-tests
**Data Science Lab (CS2311)**

**The question every t-test answers:** "Is the difference between two averages real, or could it just be random noise?"

**Intuition — signal vs noise:**

    t = (difference between the means) / (standard error of that difference)

- Big |t| means the difference is large compared with the natural scatter in the data, so it is probably real.
- **p-value:** if there were truly NO difference, the probability of getting a t at least this extreme just by chance.

**Decision rule (write this in every answer):**
- H0 (null hypothesis): the two means are equal.
- H1 (alternative): the two means are different (two-tailed test).
- If p < alpha (0.05) → **reject H0** → the difference is statistically significant.
- If p ≥ alpha → **fail to reject H0** (never write "accept H0").

| Situation | Test | SciPy function |
|---|---|---|
| Two separate groups of different people | Independent two-sample t-test | `stats.ttest_ind(a, b)` |
| Same people measured twice (before/after) | Paired t-test | `stats.ttest_rel(after, before)` |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')
ALPHA = 0.05


def decide(p, alpha=ALPHA):
    """Turn a p-value into the standard conclusion sentence."""
    if p < alpha:
        return f'p = {p:.6g} < {alpha} → Reject H0: the difference IS statistically significant.'
    return f'p = {p:.6g} ≥ {alpha} → Fail to reject H0: NO statistically significant difference.'

---
## Question 1 — Independent two-sample t-test (two teaching methods)

Group A and Group B are **different students**, so the samples are independent.

**Hypotheses:**
- H0: mean score (method A) = mean score (method B)
- H1: mean score (method A) ≠ mean score (method B)

In [ ]:
group_A = [65, 70, 68, 72, 66, 75, 69, 71, 67, 70]
group_B = [72, 75, 78, 74, 80, 77, 73, 79, 76, 81]

summary = pd.DataFrame({
    'Group A': [np.mean(group_A), np.std(group_A, ddof=1), len(group_A)],
    'Group B': [np.mean(group_B), np.std(group_B, ddof=1), len(group_B)],
}, index=['Mean', 'Std dev (sample)', 'n'])
summary.round(3)

### Checking the assumptions (good practice, earns marks)

The independent t-test assumes:
1. **Each group is roughly normal** → Shapiro–Wilk test (p > 0.05 means "no evidence against normality").
2. **Equal variances** (for the classic Student's version) → Levene's test (p > 0.05 means "variances look equal").

If Levene's test fails, use **Welch's t-test** (`equal_var=False`), which does not assume equal variances.

In [ ]:
print('Shapiro-Wilk  A: p =', round(stats.shapiro(group_A).pvalue, 4))
print('Shapiro-Wilk  B: p =', round(stats.shapiro(group_B).pvalue, 4))
lev = stats.levene(group_A, group_B)
print('Levene (equal variances): p =', round(lev.pvalue, 4))
equal_var = lev.pvalue > ALPHA
print('Use equal_var =', equal_var)

In [ ]:
t_stat, p_value = stats.ttest_ind(group_A, group_B, equal_var=equal_var)
print(f't-statistic = {t_stat:.4f}')
print(f'p-value     = {p_value:.8f}')
print(f'degrees of freedom = {len(group_A) + len(group_B) - 2}')
print(decide(p_value))

### Doing it by hand (to understand the formula)

    pooled variance  sp² = [ (n1−1)·s1² + (n2−1)·s2² ] / (n1 + n2 − 2)
    standard error   SE  = sqrt( sp² · (1/n1 + 1/n2) )
    t                    = (mean1 − mean2) / SE

The cell below computes it manually; it must match SciPy.

In [ ]:
a, b = np.array(group_A), np.array(group_B)
n1, n2 = len(a), len(b)
sp2 = ((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1)) / (n1 + n2 - 2)
se = np.sqrt(sp2 * (1 / n1 + 1 / n2))
t_manual = (a.mean() - b.mean()) / se
p_manual = 2 * stats.t.sf(abs(t_manual), df=n1 + n2 - 2)   # two-tailed
print(f'mean A = {a.mean()}, mean B = {b.mean()}, difference = {a.mean() - b.mean():.2f}')
print(f'pooled variance = {sp2:.4f}, SE = {se:.4f}')
print(f'manual t = {t_manual:.4f}, manual p = {p_manual:.8f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=pd.DataFrame({'Method A': group_A, 'Method B': group_B}), ax=ax[0])
ax[0].set_title('Score distribution by teaching method')
ax[0].set_ylabel('Exam score')
ax[1].bar(['Method A', 'Method B'], [a.mean(), b.mean()],
          yerr=[a.std(ddof=1), b.std(ddof=1)], capsize=8, color=['tab:blue', 'tab:orange'])
ax[1].set_ylim(60, 85)
ax[1].set_title('Mean score ± 1 std dev')
plt.tight_layout()
plt.show()

### Conclusion (Q1)
Method B's mean (76.5) is 7.2 points higher than method A's (69.3). The t-statistic is about −5.36 with p ≈ 0.00004, far below 0.05, so we **reject H0**: the two teaching methods produce significantly different scores, with method B performing better. (The negative sign only means group A was passed first and has the smaller mean.)

---
## Question 2 — Paired t-test (before vs after training)

The **same students** are measured twice, so each "before" score is linked to one "after" score. We must use the **paired** test.

**Key idea:** a paired t-test is just a one-sample t-test on the **differences** (after − before), testing whether their mean is 0.

**Hypotheses:**
- H0: mean difference = 0 (training had no effect)
- H1: mean difference ≠ 0

In [ ]:
before = [52, 48, 65, 60, 55, 70, 58, 62, 50, 67]
after  = [60, 55, 70, 66, 63, 75, 64, 68, 57, 72]

paired_df = pd.DataFrame({'before': before, 'after': after})
paired_df['difference'] = paired_df['after'] - paired_df['before']
paired_df

In [ ]:
d = paired_df['difference']
print('Mean difference      =', d.mean())
print('Std of differences   =', round(d.std(ddof=1), 4))
print('Shapiro-Wilk on differences: p =', round(stats.shapiro(d).pvalue, 4),
      '(paired test assumes the DIFFERENCES are roughly normal)')

t_stat, p_value = stats.ttest_rel(after, before)
print(f'\nt-statistic = {t_stat:.4f}')
print(f'p-value     = {p_value:.10f}')
print(f'degrees of freedom = {len(before) - 1}')
print(decide(p_value))

### Manual dry run

    differences d = after − before = 8, 7, 5, 6, 8, 5, 6, 6, 7, 5
    mean d̄       = 63 / 10 = 6.3
    s_d           = sqrt( Σ(d − d̄)² / (n − 1) ) = sqrt(12.1 / 9) = 1.159
    SE            = s_d / sqrt(n) = 1.159 / 3.162 = 0.367
    t             = d̄ / SE = 6.3 / 0.367 ≈ 17.18,  df = 9

In [ ]:
n = len(d)
t_manual = d.mean() / (d.std(ddof=1) / np.sqrt(n))
p_manual = 2 * stats.t.sf(abs(t_manual), df=n - 1)
print(f'manual t = {t_manual:.4f}, manual p = {p_manual:.3e}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for i in range(len(before)):
    ax[0].plot(['Before', 'After'], [before[i], after[i]], marker='o', color='gray', alpha=0.7)
ax[0].plot(['Before', 'After'], [np.mean(before), np.mean(after)], marker='o', color='red', lw=3, label='Mean')
ax[0].set_title('Each line = one student')
ax[0].set_ylabel('Programming score')
ax[0].legend()
ax[1].bar(range(1, n + 1), d, color='tab:green')
ax[1].axhline(d.mean(), color='red', ls='--', label=f'Mean improvement = {d.mean():.1f}')
ax[1].set_xlabel('Student')
ax[1].set_ylabel('After − Before')
ax[1].set_title('Improvement per student')
ax[1].legend()
plt.tight_layout()
plt.show()

### Conclusion (Q2)
Every student improved (by 5 to 8 points, mean 6.3). The paired t-test gives t ≈ 17.18 with p ≈ 3.5 × 10⁻⁸, far below 0.05, so we **reject H0**: the training program produced a statistically significant increase in programming scores.

---
## Question 3 — Independent vs paired: why do they give different results?

**Dataset of our choice:** systolic blood pressure (mmHg) of 12 patients measured **before** and **after** 8 weeks on a new medication. These are illustrative numbers created for this demonstration.

We run **both** tests on the same numbers:
- The **paired test** is the correct one (same patients twice).
- The **independent test** wrongly pretends the "before" and "after" groups are different people.

To show the opposite situation too, we also create a genuinely independent dataset (two different groups of patients), where only the independent test makes sense.

In [ ]:
bp_before = [150, 142, 165, 138, 158, 147, 170, 155, 144, 161, 149, 153]
bp_after  = [144, 139, 158, 135, 151, 143, 162, 150, 140, 155, 146, 147]

bp = pd.DataFrame({'before': bp_before, 'after': bp_after})
bp['difference'] = bp['after'] - bp['before']
print(bp.describe().round(2).loc[['mean', 'std']])

t_ind, p_ind = stats.ttest_ind(bp_after, bp_before)   # WRONG for this design
t_rel, p_rel = stats.ttest_rel(bp_after, bp_before)   # CORRECT for this design

results = pd.DataFrame({
    'Test': ['Independent t-test', 'Paired t-test'],
    't-statistic': [t_ind, t_rel],
    'p-value': [p_ind, p_rel],
    'Decision (alpha = 0.05)': ['Reject H0' if p_ind < ALPHA else 'Fail to reject H0',
                                'Reject H0' if p_rel < ALPHA else 'Fail to reject H0'],
})
results

### Same thing on the Q2 data (programming scores)

In [ ]:
t_ind2, p_ind2 = stats.ttest_ind(after, before)
t_rel2, p_rel2 = stats.ttest_rel(after, before)
pd.DataFrame({
    'Test': ['Independent t-test', 'Paired t-test'],
    't-statistic': [t_ind2, t_rel2],
    'p-value': [p_ind2, p_rel2],
    'Decision (alpha = 0.05)': ['Reject H0' if p_ind2 < ALPHA else 'Fail to reject H0',
                                'Reject H0' if p_rel2 < ALPHA else 'Fail to reject H0'],
})

### Where the difference comes from: the noise term

Both tests have the **same signal** (the mean change). They differ only in the **noise** they divide by:

- **Independent test noise:** built from the spread of scores **between people** (some patients naturally have 140, others 170).
- **Paired test noise:** built from the spread of **each person's own change** (everyone dropped by roughly 3–8 mmHg).

The cell below prints both standard errors side by side.

In [ ]:
def standard_errors(x_after, x_before):
    x_after, x_before = np.array(x_after), np.array(x_before)
    n1, n2 = len(x_after), len(x_before)
    sp2 = ((n1 - 1) * x_after.var(ddof=1) + (n2 - 1) * x_before.var(ddof=1)) / (n1 + n2 - 2)
    se_ind = np.sqrt(sp2 * (1 / n1 + 1 / n2))
    diff = x_after - x_before
    se_rel = diff.std(ddof=1) / np.sqrt(len(diff))
    return diff.mean(), se_ind, se_rel

for name, (x_a, x_b) in {'Blood pressure': (bp_after, bp_before),
                         'Programming scores': (after, before)}.items():
    m, se_i, se_r = standard_errors(x_a, x_b)
    print(f'{name:20s} mean change = {m:6.2f} | SE independent = {se_i:5.2f} | SE paired = {se_r:5.2f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist([bp_before, bp_after], bins=6, label=['Before', 'After'], alpha=0.8)
ax[0].set_title('What the INDEPENDENT test sees:\ntwo heavily overlapping groups')
ax[0].set_xlabel('Systolic BP (mmHg)')
ax[0].legend()
ax[1].hist(bp['difference'], bins=6, color='tab:green')
ax[1].axvline(0, color='red', ls='--', label='No change')
ax[1].set_title('What the PAIRED test sees:\nevery change is below zero')
ax[1].set_xlabel('After − Before (mmHg)')
ax[1].legend()
plt.tight_layout()
plt.show()

### A genuinely independent design
Two **different** groups of patients: one on the new drug, one on a placebo. Here there is no natural pairing (patient 1 in group 1 has nothing to do with patient 1 in group 2), so only the independent test is valid.

In [ ]:
drug    = [128, 135, 131, 126, 138, 133, 129, 136, 130, 132]
placebo = [139, 145, 136, 142, 148, 140, 137, 144, 141, 146]
t, p = stats.ttest_ind(drug, placebo)
print(f'Independent t-test (drug vs placebo): t = {t:.3f}, p = {p:.6f}')
print(decide(p))

### Conclusion (Q3)

- On paired data, the **paired test** gave a much larger |t| and a much smaller p-value than the independent test, even though the mean change was identical.
- **Reason:** the independent test treats the large person-to-person variation (people simply have different baseline BP or skill) as noise. The paired test removes it by looking only at each person's own change, which is very consistent. Smaller standard error → bigger t → smaller p.
- With the programming data, this changes the conclusion: the independent test is **not significant** (p ≈ 0.059) while the paired test is **highly significant** (p ≈ 3.5 × 10⁻⁸). Using the wrong test would wrongly conclude the training had no effect.
- **Rule:** same subjects measured twice → paired t-test. Different subjects in each group → independent t-test.